In [1]:
from pathlib import Path
import sys

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "requirements.txt").is_file()
            and (candidate / "libs").is_dir()
        ):
            return candidate

    raise RuntimeError("Project root could not be found.")

ROOT = find_project_root()

DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"

REPORTS_DIR = RESULTS_DIR / "reports"
CURVES_DIR = RESULTS_DIR / "curves_data"
AGGREGATED_REPORTS_DIR = RESULTS_DIR / "aggregated_reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CURVES_DIR.mkdir(parents=True, exist_ok=True)
AGGREGATED_REPORTS_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
import os
import glob
import pandas as pd

def agregar_relatorios():
    pasta_origem = REPORTS_DIR
    pasta_destino = AGGREGATED_REPORTS_DIR
    
    # Checks whether the source folder exists
    if not os.path.exists(pasta_origem):
        print(f"❌ Error: The folder '{pasta_origem}' was not found!")
        return

    # Creates the destination folder if it does not exist
    os.makedirs(pasta_destino, exist_ok=True)
    
    # Gets all CSV files in the folder
    ficheiros_csv = glob.glob(os.path.join(pasta_origem, "*.csv"))
    
    if not ficheiros_csv:
        print(f"⚠️ No CSV files found in folder '{pasta_origem}'.")
        return

    print(f"Found {len(ficheiros_csv)} files. Starting aggregation...\n")
    
    # Dictionary used to store grouped DataFrames
    # The key will be the tuple: (Algorithm, Dataset, Mode)
    agrupamentos = {}
    
    for ficheiro in ficheiros_csv:
        # Extracts the file name without the path or extension
        nome_ficheiro = os.path.basename(ficheiro).replace(".csv", "")
        
        # Our naming convention is: algorithm_dataset_mode_binarization
        # Example 1: standart_wisard_UNSW-NB15_binary_distributive
        # Example 2: decision_tree_Bot-IoT_multiclass
        
        partes = nome_ficheiro.split('_')
        
        # Logic used to identify the parts based on the filename structure
        if len(partes) >= 4:
            if partes[0] == "decision" and partes[1] == "tree":
                algoritmo = "decision_tree"
                dataset = partes[2]
                modo = partes[3]
                binarizacao = "N/A" # Trees do not use thermometer encoding
            elif partes[0] == "random" and partes[1] == "forest":
                algoritmo = "random_forest"
                dataset = partes[2]
                modo = partes[3]
                binarizacao = "N/A"
            else:
                # It is a WiSARD model (standart_wisard or bloom_wisard)
                algoritmo = f"{partes[0]}_{partes[1]}"
                dataset = partes[2]
                modo = partes[3]
                binarizacao = partes[4] if len(partes) > 4 else "N/A"
        else:
            print(f"⚠️ Warning: File with a non-standard name ignored: {nome_ficheiro}")
            continue
            
        chave_grupo = (algoritmo, dataset, modo)
        
        # Reads the current CSV file
        try:
            df_temp = pd.read_csv(ficheiro)
            # Adds the binarization column so the data source can be identified
            if "Binarization" not in df_temp.columns:
                # Insert at the beginning of the DataFrame
                df_temp.insert(2, "Binarization", binarizacao) 
            
            # Stores it in the grouping list
            if chave_grupo not in agrupamentos:
                agrupamentos[chave_grupo] = []
            agrupamentos[chave_grupo].append(df_temp)
            
        except Exception as e:
            print(f"❌ Error reading {ficheiro}: {e}")

    # Now merge and save the grouped files
    for (algoritmo, dataset, modo), lista_dfs in agrupamentos.items():
        # Concatenates all DataFrames in this group
        df_final = pd.concat(lista_dfs, ignore_index=True)
        
        # Final aggregated filename
        nome_saida = f"{algoritmo}_{dataset}_{modo}_COMPLETO.csv"
        caminho_saida = os.path.join(pasta_destino, nome_saida)
        
       # Saves to disk
        # sep=';' separates columns correctly for Portuguese-locale Excel
        # decimal=',' replaces decimal points with commas in numeric values
        df_final.to_csv(caminho_saida, index=False, sep=';', decimal=',')
        
        print(f"✅ Aggregated successfully: {nome_saida} (Total rows: {len(df_final)})")

    print("\n🚀 AGGREGATION PROCESS COMPLETED!")

if __name__ == "__main__":
    agregar_relatorios()

Found 53 files. Starting aggregation...

✅ Aggregated successfully: bloom_wisard_Bot-IoT_binary_COMPLETO.csv (Total rows: 1488)
✅ Aggregated successfully: bloom_wisard_Bot-IoT_multiclass_COMPLETO.csv (Total rows: 1008)
✅ Aggregated successfully: bloom_wisard_CICIDS_binary_COMPLETO.csv (Total rows: 1296)
✅ Aggregated successfully: bloom_wisard_CICIDS_multiclass_COMPLETO.csv (Total rows: 806)
✅ Aggregated successfully: bloom_wisard_Edge-IIoT_binary_COMPLETO.csv (Total rows: 1440)
✅ Aggregated successfully: bloom_wisard_ToN-IoT_binary_COMPLETO.csv (Total rows: 1440)
✅ Aggregated successfully: bloom_wisard_UNSW-NB15_binary_COMPLETO.csv (Total rows: 1504)
✅ Aggregated successfully: bloom_wisard_UNSW-NB15_multiclass_COMPLETO.csv (Total rows: 1008)
✅ Aggregated successfully: cw_generation_stats_Bot-IoT_COMPLETO.csv (Total rows: 2)
✅ Aggregated successfully: cw_generation_stats_CICIDS_COMPLETO.csv (Total rows: 2)
✅ Aggregated successfully: cw_generation_stats_UNSW-NB15_COMPLETO.csv (Total rows